[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hudson11111/MMA3001Project/blob/main/notebooks/00_colab_setup_and_first_look.ipynb)

# 00 — Colab setup and first look at the data

**Project:** Automated inspection of pork-rasher packaging (MMA3001)
**Work package:** WP0 (environment + data access) — see `docs/PROJECT_PLAN.md`

This notebook:
1. records the environment (Python, GPU, package versions) for reproducibility,
2. gets the project code from GitHub,
3. downloads Roboflow dataset **version 4** using your API key from Colab Secrets,
4. checks the download against the numbers published on Roboflow,
5. gives a first look at class balance, defect-free ("null") images and examples,
6. saves a summary JSON and a small development sample.

**Before running:** add a Colab secret named `ROBOFLOW_API_KEY` (key icon in the left
sidebar → *Add new secret* → paste key → switch **Notebook access** on).
A CPU runtime is fine for this notebook.


## 1. Environment record

In [ ]:
import sys, platform, json, datetime

env_info = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
}
try:
    import torch
    env_info["torch"] = torch.__version__
    env_info["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        env_info["gpu"] = torch.cuda.get_device_name(0)
except ImportError:
    env_info["torch"] = None
print(json.dumps(env_info, indent=2))

## 2. Get the project code

The notebook imports the tested `porkqc` package instead of defining functions inline
(DECISION_LOG D-002). We add `src/` to `sys.path` rather than `pip install -e .`,
because Colab only picks up editable installs after a runtime restart.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Hudson11111/MMA3001Project.git"
REPO_DIR = "/content/MMA3001Project"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

import porkqc
env_info["porkqc"] = porkqc.__version__
env_info["git_commit"] = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
for pkg in ("roboflow", "ultralytics"):
    try:
        env_info[pkg] = __import__(pkg).__version__
    except Exception:
        env_info[pkg] = None
print(json.dumps(env_info, indent=2))

## 3. (Optional) Mount Google Drive

Colab's disk is wiped when the runtime disconnects. Later notebooks save training runs
and weights to Drive. Set `USE_DRIVE = False` to skip.

In [ ]:
from pathlib import Path
from porkqc.config import load_yaml

cfg = load_yaml("configs/dataset.yaml")
USE_DRIVE = True

DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path(cfg["paths"]["drive_project_dir"])
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive project folder:", DRIVE_DIR)

## 4. Download the dataset (Roboflow v4, YOLO format)

The API key is read from Colab Secrets by `porkqc.env.get_secret` and is never printed.
Export format `yolov8` has the same layout as the YOLO11/YOLO26 exports (D-015).

In [ ]:
from porkqc.data.download import download_dataset

rf_cfg = cfg["roboflow"]
data_root = download_dataset(
    workspace=rf_cfg["workspace"],
    project=rf_cfg["project"],
    version=rf_cfg["version"],
    location=cfg["paths"]["colab_data_dir"],
    export_format=rf_cfg["export_format"],
)
print("Dataset folder:", data_root)
!ls {data_root}

### 4.1 Repair `data.yaml` and check class names

Roboflow often writes relative split paths (`../train/images`) that break training.
`fix_data_yaml` rewrites them as absolute paths (original kept as `data.yaml.orig`).
The class-name check stops the notebook if class ids would mean something different
from what the code assumes.

In [ ]:
from porkqc import CLASS_NAMES
from porkqc.config import check_class_names
from porkqc.data.download import fix_data_yaml, read_class_names

yaml_path = Path(data_root) / "data.yaml"
print("--- data.yaml as downloaded ---")
print(yaml_path.read_text())

fixed = fix_data_yaml(yaml_path)
names = read_class_names(yaml_path)
check_class_names(names, CLASS_NAMES)
print("Class names OK:", names)
print({k: fixed[k] for k in ("train", "val", "test")})

## 5. Check the download against the published numbers

Roboflow lists 3,349 train / 120 valid / 80 test images. We also count *unique
originals* (Roboflow file names keep the source image name before `.rf.`), which tells
us how many real photographs sit behind the augmented training copies.

In [ ]:
from porkqc.data.audit import summarise_dataset

summaries = summarise_dataset(data_root, n_classes=len(CLASS_NAMES))
expected = cfg["expected"]["split_counts"]
for split, s in summaries.items():
    flag = "OK" if s.n_images == expected[split] else "MISMATCH <-- investigate"
    print(f"{split:5s}: {s.n_images:5d} images (expected {expected[split]:4d}) {flag}"
          f" | unique originals: {s.n_unique_originals}")

## 6. First look: labels, null images and class balance

Key questions (PROJECT_PLAN §2):
* **Null images** — are there trays with *no* defects? Without them a PASS decision
  cannot be validated.
* **Class balance** — rare classes will have very uncertain per-class scores,
  especially in the 80-image test split.
* **Label problems** — any missing or malformed label files.

In [ ]:
import pandas as pd

overview = pd.DataFrame([
    {
        "split": split,
        "images": s.n_images,
        "unique originals": s.n_unique_originals,
        "null (no-defect) images": s.n_null_images,
        "missing label files": s.n_missing_labels,
        "invalid label files": s.n_invalid_label_files,
        "objects": s.n_instances,
        "max objects / image": s.boxes_per_image_max,
    }
    for split, s in summaries.items()
]).set_index("split")
display(overview)

per_class = pd.DataFrame({split: s.instances_per_class for split, s in summaries.items()}, index=CLASS_NAMES)
per_class["total"] = per_class.sum(axis=1)
print("Objects per class")
display(per_class)

img_per_class = pd.DataFrame({split: s.images_per_class for split, s in summaries.items()}, index=CLASS_NAMES)
print("Images containing each class")
display(img_per_class)

for split, s in summaries.items():
    if s.issues:
        print(f"--- {split}: first issues ---")
        print("\n".join(s.issues[:10]))

### 6.1 Image sizes and colour modes

The I/O specification assumes 720 × 540 RGB. (This check moves into
`porkqc.data.audit` in WP1.)

In [ ]:
from collections import Counter
from PIL import Image
from porkqc.data.audit import list_images

size_counts = Counter()
for split in summaries:
    for p in list_images(Path(data_root) / split / "images"):
        with Image.open(p) as im:
            size_counts[(split, im.size, im.mode)] += 1

sizes_df = pd.DataFrame(
    [{"split": k[0], "size (W, H)": str(k[1]), "mode": k[2], "count": v} for k, v in size_counts.items()]
)
display(sizes_df)

### 6.2 Example images for each class

Looking at real examples is the quickest way to understand what each label means —
for instance, how `packaging-error` differs from `unsealed`. One example per original
image (augmented copies skipped).

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from porkqc.data.audit import original_stem
from porkqc.data.labels import read_label_file, yolo_to_xyxy

SPLIT, N_PER_CLASS, SEED = "train", 3, 0
rng = random.Random(SEED)
imgs = list_images(Path(data_root) / SPLIT / "images")
rng.shuffle(imgs)

examples = {c: [] for c in range(len(CLASS_NAMES))}
seen_originals = set()
for p in imgs:
    stem = original_stem(p)
    if stem in seen_originals:
        continue
    lab_path = Path(data_root) / SPLIT / "labels" / f"{p.stem}.txt"
    if not lab_path.exists():
        continue
    labels = read_label_file(lab_path, len(CLASS_NAMES))
    for c in set(labels[:, 0].astype(int).tolist()):
        if len(examples[c]) < N_PER_CLASS:
            examples[c].append((p, labels))
            seen_originals.add(stem)
    if all(len(v) >= N_PER_CLASS for v in examples.values()):
        break

colours = plt.cm.tab10.colors
fig, axes = plt.subplots(len(CLASS_NAMES), N_PER_CLASS, figsize=(4.2 * N_PER_CLASS, 3.3 * len(CLASS_NAMES)))
for c, row in enumerate(axes):
    for j, ax in enumerate(row):
        ax.axis("off")
        if j >= len(examples[c]):
            continue
        p, labels = examples[c][j]
        with Image.open(p) as im:
            ax.imshow(im.convert("RGB"))
            w, h = im.size
        for (cls, *_), (x1, y1, x2, y2) in zip(labels, yolo_to_xyxy(labels, w, h)):
            k = int(cls)
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                           lw=2.5 if k == c else 1, ec=colours[k]))
            ax.text(x1, y1 - 3, CLASS_NAMES[k], color=colours[k], fontsize=7)
        ax.set_title(f"{CLASS_NAMES[c]} — {p.name[:28]}", fontsize=8)
fig.tight_layout()
Path("results/figures").mkdir(parents=True, exist_ok=True)
fig.savefig("results/figures/00_examples_per_class.png", dpi=110)
plt.show()

## 7. Save the summary

In [ ]:
summary = {
    "environment": env_info,
    "dataset": {**rf_cfg, "local_path": str(data_root)},
    "class_names": list(CLASS_NAMES),
    "splits": {k: v.to_dict() for k, v in summaries.items()},
    "image_sizes": [{"split": k[0], "size": list(k[1]), "mode": k[2], "count": v} for k, v in size_counts.items()],
}
Path("results").mkdir(exist_ok=True)
with open("results/00_first_look.json", "w") as fh:
    json.dump(summary, fh, indent=2)
if DRIVE_DIR:
    (DRIVE_DIR / "results").mkdir(exist_ok=True)
    !cp results/00_first_look.json results/figures/00_examples_per_class.png {DRIVE_DIR}/results/
print(json.dumps(summary, indent=1)[:4000])

## 8. Development sample for Claude

Claude's workspace cannot reach Roboflow, so this writes a small, reproducible subset
(40 train / 20 valid / 20 test images with labels, every class represented) to a zip.
Download it and attach it in the Claude chat.

In [ ]:
from porkqc.data.audit import make_dev_sample

zip_path = make_dev_sample(data_root, "/content/dev_sample.zip", CLASS_NAMES, seed=0)
print(f"{zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)")
if DRIVE_DIR:
    !cp {zip_path} {DRIVE_DIR}/
from google.colab import files
files.download(str(zip_path))

## 9. Hand back to Claude

Send Claude **either**:
* *File → Save a copy in GitHub* → repository `Hudson11111/MMA3001Project`, branch `main`,
  path `notebooks/00_colab_setup_and_first_look.ipynb` (saves this notebook *with* its outputs), **or**
* attach `results/00_first_look.json`, `results/figures/00_examples_per_class.png`
  (Files panel on the left) and `dev_sample.zip` to the chat.

Anything that printed `MISMATCH`, an error, or surprised you is worth mentioning too.